# Analyze a Document with a Custom Document Intelligence Model

Run these cells one at a time to test the custom extraction model you trained in Document Intelligence Studio. This notebook uses `DefaultAzureCredential`, so it runs with your current Azure user credentials.

## Before You Start

Make sure `workshop/.env` exists and includes `DOC_INTELLIGENCE_ENDPOINT` and `DOC_INTELLIGENCE_MODEL_ID`. Sign in with an Azure identity that has access to the Document Intelligence resource before running the analysis cells.

In [8]:
from pathlib import Path
import os

from IPython.display import Markdown, display
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest

## Load Configuration

This cell finds `workshop/.env` from either the repository root or the notebook folder, then reads the Document Intelligence endpoint and custom model ID.

In [9]:
def find_workshop_env() -> Path:
    start = Path.cwd().resolve()
    candidates = [start / 'workshop' / '.env', start / '.env']

    for parent in start.parents:
        candidates.extend([parent / 'workshop' / '.env', parent / '.env'])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError('Could not find workshop/.env. Copy workshop/.env.sample to workshop/.env first.')


env_path = find_workshop_env()
load_dotenv(env_path)

endpoint = (os.getenv('DOC_INTELLIGENCE_ENDPOINT') or '').strip()
model_id = (os.getenv('DOC_INTELLIGENCE_MODEL_ID') or '').strip()

if not endpoint:
    raise ValueError('DOC_INTELLIGENCE_ENDPOINT is not set in workshop/.env.')
if not model_id or model_id.startswith('<'):
    raise ValueError('DOC_INTELLIGENCE_MODEL_ID is not set in workshop/.env. Train a custom model and paste its model ID first.')

print(f'Loaded settings from: {env_path}')
print(f'Document Intelligence endpoint: {endpoint}')
print(f'Custom model ID: {model_id}')

Loaded settings from: C:\Users\algut\repos\alesaez\content-processing-solution\workshop\.env
Document Intelligence endpoint: https://docintelltecjexcelaiai.cognitiveservices.azure.com/
Custom model ID: custom-model-invoice


## Set the Test Document

The script uses the Microsoft Learn sample form URL. The same test image is also included in this lab folder as `custom/test1.jpg` for inspection.

In [10]:
form_url = 'https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/03-document-intelligence/custom/test1.jpg?raw=true'

print(f'Analyzing form at: {form_url}')
print(f'Using custom model: {model_id}')

Analyzing form at: https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/03-document-intelligence/custom/test1.jpg?raw=true
Using custom model: custom-model-invoice


## Create the Client

The custom model flow uses `DefaultAzureCredential`, which can use your current Azure CLI, VS Code, managed identity, or other configured Azure identity.

In [11]:
credential = DefaultAzureCredential()
document_analysis_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=credential
)

print('Document Intelligence client created.')

Document Intelligence client created.


## Analyze the Form

This cell sends the form URL to your custom model and waits for the analysis operation to complete.

In [12]:
poller = document_analysis_client.begin_analyze_document(
    model_id,
    AnalyzeDocumentRequest(url_source=form_url)
)

result = poller.result()
print('Analysis complete.')

Analysis complete.


## Review Extracted Fields

This cell displays every field returned by the custom model in a table, including the extracted value and confidence score.

In [13]:
def escape_markdown_table_value(value) -> str:
    text = 'N/A' if value is None else str(value)
    return text.replace('\\', '\\\\').replace('|', '\\|').replace('\n', '<br>')


field_rows = []
for document_index, document in enumerate(result.documents, start=1):
    for name, field in document.fields.items():
        field_value = field.get('valueString') or field.get('content', 'N/A')
        field_rows.append({
            'Document': document_index,
            'Document Type': document.doc_type,
            'Document Confidence': document.confidence,
            'Model ID': result.model_id,
            'Field': name,
            'Value': field_value,
            'Field Confidence': field.get('confidence'),
        })

if not field_rows:
    display(Markdown('No fields were returned by the custom model.'))
else:
    headers = ['Document', 'Document Type', 'Document Confidence', 'Model ID', 'Field', 'Value', 'Field Confidence']
    table_lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in field_rows:
        table_lines.append('| ' + ' | '.join(escape_markdown_table_value(row[header]) for header in headers) + ' |')

    display(Markdown('\n'.join(table_lines)))

| Document | Document Type | Document Confidence | Model ID | Field | Value | Field Confidence |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | PurchaseOrderNumber | 3929423 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Signature | Josh Granger | 0.624 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Tax | $600.00 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Merchant | Hero Limited | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | PhoneNumber | 555-348-6512 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Website | www.herolimited.com | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | DatedAs | 04/04/2020 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Total | $7350.00 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Subtotal | $6750.00 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | CompanyName | Yoga for You | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | CompanyPhoneNumber | 234-986-6454 | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | CompanyAddress | 343 E Winter Road Seattle, WA 93849 Phone: | 0.604 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Email | accounts@herolimited.com | 0.853 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | VendorName | Seth Stanley | 0.99 |
| 1 | custom-model-invoice:custom-model-invoice | 0.818 | custom-model-invoice | Quantity | 50 | 0.99 |

## Optional: Inspect Field Objects

Run this cell if you want to inspect each returned field object more closely.

In [14]:
for document_index, document in enumerate(result.documents, start=1):
    print(f'Document {document_index}')
    for field_name, field in document.fields.items():
        print(f'{field_name}:')
        print(field)
        print()

Document 1
PurchaseOrderNumber:
{'type': 'string', 'valueString': '3929423', 'content': '3929423', 'boundingRegions': [{'pageNumber': 1, 'polygon': [1282, 459, 1391, 459, 1391, 490, 1282, 490]}], 'confidence': 0.99, 'spans': [{'offset': 204, 'length': 7}]}

Signature:
{'type': 'string', 'valueString': 'Josh Granger', 'content': 'Josh Granger', 'boundingRegions': [{'pageNumber': 1, 'polygon': [422, 1666, 626, 1666, 626, 1704, 422, 1704]}], 'confidence': 0.624, 'spans': [{'offset': 703, 'length': 12}]}

Tax:
{'type': 'string', 'valueString': '$600.00', 'content': '$600.00', 'boundingRegions': [{'pageNumber': 1, 'polygon': [1428, 1614, 1531, 1614, 1531, 1645, 1428, 1645]}], 'confidence': 0.99, 'spans': [{'offset': 680, 'length': 7}]}

Merchant:
{'type': 'string', 'valueString': 'Hero Limited', 'content': 'Hero Limited', 'boundingRegions': [{'pageNumber': 1, 'polygon': [616, 201, 1075, 201, 1075, 269, 616, 269]}], 'confidence': 0.99, 'spans': [{'offset': 15, 'length': 12}]}

PhoneNumber:
{